# Day 7: CRUD Mastery - Pagination, Filtering, Sorting, and Bulk Operations

## Goal
Build production-ready CRUD endpoints that handle real-world requirements: pagination, filtering, sorting, partial updates, and error handling.

## Why This Day Matters
Real APIs never return 10,000 records at once. You'll learn patterns that scale from startups to enterprises.

In [ ]:
from fastapi import FastAPI, Query, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

app = FastAPI(title="CRUD Mastery API")

print("✓ FastAPI + pagination patterns")

## PART 1: PAGINATION PATTERNS

Return data in pages instead of all at once.

In [ ]:
# EXAMPLE 1: Basic pagination
class PaginationParams(BaseModel):
    skip: int = Query(0, ge=0, description="Skip N records")
    limit: int = Query(10, ge=1, le=100, description="Return at most N records")

class PaginatedResponse(BaseModel):
    data: list
    total: int
    skip: int
    limit: int
    has_more: bool

print("✓ Pagination structure:")
print("  skip: number of records to skip (pagination position)")
print("  limit: maximum records per page")
print("  total: total count in database")
print("  has_more: whether more records exist after this page")

In [ ]:
# EXAMPLE 2: Implementing pagination in endpoint pseudocode
example_endpoint = '''
@app.get("/items", response_model=PaginatedResponse)
def list_items(
    skip: int = Query(0, ge=0),
    limit: int = Query(10, ge=1, le=100),
    db: Session = Depends(get_db)
):
    # Get total count
    total = db.query(Item).count()
    
    # Apply offset and limit
    items = db.query(Item)\
        .offset(skip)\
        .limit(limit)\
        .all()
    
    return PaginatedResponse(
        data=items,
        total=total,
        skip=skip,
        limit=limit,
        has_more=(skip + limit) < total
    )
'''

print("✓ Pagination endpoint pattern:")
print(example_endpoint)

## PART 2: FILTERING

Let users query specific data.

In [ ]:
# EXAMPLE: Multi-field filtering
example_filtering = '''
@app.get("/products")
def list_products(
    category: Optional[str] = Query(None),           # Filter by category
    min_price: float = Query(0, ge=0),               # Price range
    max_price: float = Query(999999, ge=0),
    in_stock: bool = Query(True),                    # Only in stock items
    search: Optional[str] = Query(None, min_length=1),  # Search name/description
    skip: int = Query(0),
    limit: int = Query(10),
    db: Session = Depends(get_db)
):
    query = db.query(Product)
    
    # Apply filters
    if category:
        query = query.filter(Product.category == category)
    
    query = query.filter(Product.price >= min_price)
    query = query.filter(Product.price <= max_price)
    
    if in_stock:
        query = query.filter(Product.stock > 0)
    
    if search:
        # Search in name or description
        search_term = f"%{search}%"
        query = query.filter(
            (Product.name.ilike(search_term)) |
            (Product.description.ilike(search_term))
        )
    
    # Pagination
    total = query.count()
    items = query.offset(skip).limit(limit).all()
    
    return {"items": items, "total": total}
'''

print("✓ Filtering allows queries like:")
print("  /products?category=electronics&min_price=100&max_price=500")
print("  /products?search=laptop&in_stock=true")

## PART 3: SORTING

Order results by different fields.

## PART 4: PARTIAL UPDATES (PATCH)

Update only changed fields.

In [ ]:
# EXAMPLE: PATCH endpoint
example_patch = '''
from pydantic import BaseModel
from typing import Optional

class ProductUpdate(BaseModel):
    """All fields optional for PATCH requests"""
    name: Optional[str] = None
    description: Optional[str] = None
    price: Optional[float] = None
    stock: Optional[int] = None

@app.patch("/products/{product_id}")
def update_product(
    product_id: int,
    update_data: ProductUpdate,
    db: Session = Depends(get_db)
):
    product = db.query(Product).filter(Product.id == product_id).first()
    if not product:
        raise HTTPException(status_code=404)
    
    # Update only provided fields
    update_dict = update_data.model_dump(exclude_unset=True)
    for field, value in update_dict.items():
        setattr(product, field, value)
    
    db.commit()
    db.refresh(product)
    return product
'''

print("✓ PATCH vs PUT:")
print("  PUT: Replace entire resource (all fields required)")
print("  PATCH: Partial update (only changed fields)")
print("\n✓ Example: PATCH /products/1")
print("  Request body: {\"price\": 29.99}")
print("  Result: Only price is updated, other fields unchanged")

## PART 5: BULK OPERATIONS

Handle multiple records efficiently.

## PART 6: OWNERSHIP AND PERMISSIONS

Users can only access their own data.

## PART 7: COMMON MISTAKES

### ❌ Mistake 1: Returning 10,000 records
Bad: `return db.query(Item).all()  # Could return millions of records`
Good: `return db.query(Item).limit(10).all()`

### ❌ Mistake 2: No ownership checks
Bad: User can delete another user's data
Good: Check `if post.user_id != current_user.id: raise 403`

### ❌ Mistake 3: N+1 queries in pagination
Bad: Loop through results and access relationships
Good: Use `joinedload()` or `select_in_load()`

### ❌ Mistake 4: Forgetting to validate pagination params
Bad: `skip: int, limit: int` (user can request 1 million records)
Good: `limit: int = Query(10, le=100)` (max 100)

### ❌ Mistake 5: Exposing ORM models directly
Bad: Returning database model with password field
Good: Use separate Pydantic response model without secrets

## Day 7 Summary

You now understand:
✅ Pagination (skip/limit pattern)
✅ Filtering (single and multi-field)
✅ Sorting (with enums for safety)
✅ Partial updates (PATCH)
✅ Bulk operations with limits
✅ User ownership checks
✅ Security patterns (403 vs 404)
✅ Common CRUD mistakes

**Next:** Day 8 - Authentication and Authorization